# Part 4: Mitigation — Making the Classifier Fairer and More Robust

**Objective:** Apply three bias mitigation techniques and measure their effect on both fairness metrics and overall accuracy.

1. **Technique 1 — Reweighing (pre-processing):** Use `aif360.Reweighing` to compute sample weights that counteract group bias, then retrain.
2. **Technique 2 — Threshold Optimization (post-processing):** Use `fairlearn.ThresholdOptimizer` with `equalized_odds` constraint.
3. **Technique 3 — Oversampling (data augmentation):** Duplicate high-black cohort training examples 3× and retrain.

In [ ]:
!pip install aif360 fairlearn transformers scikit-learn torch matplotlib seaborn --quiet

In [ ]:
import inspect
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
warnings.filterwarnings('ignore')

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from torch.utils.data import Dataset
from aif360.datasets import BinaryLabelDataset, StandardDataset
from aif360.algorithms.preprocessing import Reweighing
from aif360.metrics import ClassificationMetric
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import equalized_odds_difference

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
THRESHOLD = 0.4
MODEL_NAME = 'distilbert-base-uncased'
TRAIN_EVAL_ARG = (
    'evaluation_strategy'
    if 'evaluation_strategy' in inspect.signature(TrainingArguments.__init__).parameters
    else 'eval_strategy'
)
print("All libraries loaded.")

## 4.0 Load Data and Baseline

In [ ]:
df_train = pd.read_csv('train_subset.csv')
df_eval  = pd.read_csv('eval_with_preds.csv')

for col in ['black', 'white']:
    df_train[col] = df_train[col].fillna(0.0)
    df_eval[col]  = df_eval[col].fillna(0.0)

# Cohort masks on eval
black_mask = df_eval['black'] >= 0.5
white_mask = (df_eval['black'] < 0.1) & (df_eval['white'] >= 0.5)

# Cohort masks on train
black_mask_train = df_train['black'] >= 0.5

print(f"Train: {df_train.shape}, Eval: {df_eval.shape}")
print(f"Eval high-black: {black_mask.sum()}, Eval reference: {white_mask.sum()}")

In [ ]:
def cohort_metrics(y_true, y_pred, y_proba=None):
    """Compute per-cohort fairness metrics."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return {'TPR': round(tpr,4), 'FPR': round(fpr,4), 'FNR': round(fnr,4)}

def aif360_metrics(df_combined):
    """Compute AIF360 SPD and EOD from a combined dataframe."""
    aif_true = BinaryLabelDataset(
        df=df_combined[['label', 'group']].rename(columns={'label':'lbl'}),
        label_names=['lbl'], protected_attribute_names=['group'],
        favorable_label=0, unfavorable_label=1
    )
    aif_pred = aif_true.copy()
    aif_pred.labels = df_combined['y_pred'].values.reshape(-1,1)
    cm_m = ClassificationMetric(
        aif_true, aif_pred,
        unprivileged_groups=[{'group': 0}],
        privileged_groups=[{'group': 1}]
    )
    return cm_m.statistical_parity_difference(), cm_m.equal_opportunity_difference()

def evaluate_model_fairness(df_eval, black_mask, white_mask, model_label, threshold=THRESHOLD):
    """Compute full fairness report for a given prediction column."""
    y_true_all = df_eval['label'].values
    y_proba_col = df_eval['y_proba'].values
    y_pred_all = (y_proba_col >= threshold).astype(int)

    f1_all = f1_score(y_true_all, y_pred_all, average='macro')

    mb = cohort_metrics(df_eval.loc[black_mask, 'label'], df_eval.loc[black_mask, 'y_proba'].apply(lambda x: int(x>=threshold)))
    mw = cohort_metrics(df_eval.loc[white_mask, 'label'], df_eval.loc[white_mask, 'y_proba'].apply(lambda x: int(x>=threshold)))

    # AIF360
    df_combined = pd.concat([
        df_eval[black_mask].assign(group=0, y_pred=lambda d: (d['y_proba']>=threshold).astype(int)),
        df_eval[white_mask].assign(group=1, y_pred=lambda d: (d['y_proba']>=threshold).astype(int)),
    ], ignore_index=True)
    spd, eod = aif360_metrics(df_combined)

    return {
        'Model': model_label,
        'Overall F1': round(f1_all, 4),
        'HB FPR': mb['FPR'],
        'Ref FPR': mw['FPR'],
        'Disparate Impact': round(mb['FPR']/mw['FPR'], 4) if mw['FPR']>0 else 'inf',
        'SPD': round(spd, 4),
        'EOD': round(eod, 4),
    }

# Baseline
baseline_result = evaluate_model_fairness(df_eval, black_mask, white_mask, 'Baseline')
results_list = [baseline_result]
print("Baseline metrics computed.")
print(pd.DataFrame([baseline_result]).to_string(index=False))

## 4.1 Technique 1: Reweighing (Pre-Processing)

In [ ]:
# Construct AIF360 BinaryLabelDataset for train
df_train['group'] = (df_train['black'] >= 0.5).astype(int)  # 1=unprivileged, 0=privileged

# AIF360 expects group: 0 = privileged (white), 1 = unprivileged (black)
# We encode: privileged = white group (black<0.1, white>=0.5), unprivileged = black group (black>=0.5)
df_train['priv_group'] = 0  # default: neither
df_train.loc[black_mask_train, 'priv_group'] = 0   # unprivileged
df_train.loc[(df_train['black'] < 0.1) & (df_train['white'] >= 0.5), 'priv_group'] = 1  # privileged

# Only rows that belong to one of the two groups
df_rw = df_train[df_train['priv_group'].isin([0, 1]) | black_mask_train].copy()
# For simplicity, use all training data with group defined as black>=0.5 vs rest
df_rw = df_train.copy()
df_rw['group_rw'] = (df_rw['black'] >= 0.5).astype(int)  # 1 = high-black

aif_train = BinaryLabelDataset(
    df=df_rw[['label','group_rw']].rename(columns={'label':'lbl'}),
    label_names=['lbl'],
    protected_attribute_names=['group_rw'],
    favorable_label=0,
    unfavorable_label=1
)

rw = Reweighing(
    unprivileged_groups=[{'group_rw': 1}],
    privileged_groups=[{'group_rw': 0}]
)
rw.fit(aif_train)
aif_train_rw = rw.transform(aif_train)
sample_weights = aif_train_rw.instance_weights

print(f"Sample weights computed. Min: {sample_weights.min():.4f}, Max: {sample_weights.max():.4f}")
print(f"High-black avg weight: {sample_weights[df_rw['group_rw']==1].mean():.4f}")
print(f"Reference avg weight:  {sample_weights[df_rw['group_rw']==0].mean():.4f}")

In [ ]:
class WeightedToxicDataset(Dataset):
    def __init__(self, texts, labels, weights, tokenizer, max_length=128):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True,
            max_length=max_length, return_tensors='pt'
        )
        self.labels  = torch.tensor(list(labels), dtype=torch.long)
        self.weights = torch.tensor(list(weights), dtype=torch.float)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        item['weights'] = self.weights[idx]
        return item

tokenizer_rw = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizing reweighed training data...")
train_ds_rw = WeightedToxicDataset(
    df_rw['comment_text'], df_rw['label'], sample_weights, tokenizer_rw
)
df_eval_clean = pd.read_csv('eval_subset.csv')

class SimpleToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=max_length, return_tensors='pt')
        self.labels = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

eval_ds_rw = SimpleToxicDataset(df_eval_clean['comment_text'], df_eval_clean['label'], tokenizer_rw)
print("Datasets ready for reweighing experiment.")

In [ ]:
import torch.nn as nn

class WeightedTrainer(Trainer):
    """Custom Trainer that applies per-sample weights to the cross-entropy loss."""
    def __init__(self, sample_weights_tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.sample_weights_tensor = sample_weights_tensor

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.pop('labels')
        weights = inputs.pop('weights', None)
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(reduction='none')
        per_example_loss = loss_fct(logits, labels)

        if weights is not None:
            weights = weights.to(per_example_loss.device)
            loss = (per_example_loss * weights).mean()
        else:
            loss = per_example_loss.mean()

        return (loss, outputs) if return_outputs else loss

model_rw = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args_rw_kwargs = dict(
    output_dir='./results_rw',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=200,
    fp16=torch.cuda.is_available(),
    seed=SEED,
 )
strategy_key = (
    'evaluation_strategy'
    if 'evaluation_strategy' in TrainingArguments.__init__.__code__.co_varnames
    else 'eval_strategy'
)
args_rw_kwargs[strategy_key] = 'epoch'
args_rw = TrainingArguments(**args_rw_kwargs)

trainer_rw = WeightedTrainer(
    sample_weights_tensor=torch.tensor(sample_weights, dtype=torch.float),
    model=model_rw, args=args_rw,
    train_dataset=train_ds_rw,
    eval_dataset=eval_ds_rw,
)

print("Training reweighed model...")
trainer_rw.train()

In [ ]:
# Get reweighed predictions
pred_rw = trainer_rw.predict(eval_ds_rw)
proba_rw = torch.softmax(torch.tensor(pred_rw.predictions), dim=-1)[:, 1].numpy()
df_eval_rw = df_eval.copy()
df_eval_rw['y_proba'] = proba_rw

result_rw = evaluate_model_fairness(df_eval_rw, black_mask, white_mask, 'Reweighing (pre-proc)')
results_list.append(result_rw)

# Save reweighed model
model_rw.save_pretrained('./model_reweighed')
tokenizer_rw.save_pretrained('./model_reweighed')
print("Reweighed model saved.")
print(pd.DataFrame([result_rw]).to_string(index=False))

## 4.2 Technique 2: Threshold Optimization (Post-Processing)

In [ ]:
# Use baseline model probabilities + fairlearn ThresholdOptimizer
# Sensitive feature: group membership (1=high-black, 0=reference)
df_eval_to = df_eval.copy()
df_eval_to['group_to'] = 0
df_eval_to.loc[black_mask, 'group_to'] = 1
df_eval_to.loc[white_mask, 'group_to'] = 0

# Only use rows that belong to one of the two cohorts for the optimizer
cohort_mask = black_mask | white_mask
df_cohort = df_eval_to[cohort_mask].copy().reset_index(drop=True)

X_proba = df_cohort['y_proba'].values.reshape(-1, 1)
y_cohort = df_cohort['label'].values
s_cohort = df_cohort['group_to'].values

# Wrap probabilities in a simple pass-through estimator
from sklearn.base import BaseEstimator, ClassifierMixin

class ProbaWrapper(BaseEstimator, ClassifierMixin):
    """Wraps pre-computed probabilities as a sklearn-compatible estimator."""
    def fit(self, X, y): 
        self.classes_ = np.array([0, 1])
        return self
    def predict(self, X): return (X[:, 0] >= THRESHOLD).astype(int)
    def predict_proba(self, X): return np.column_stack([1-X[:,0], X[:,0]])

wrapper = ProbaWrapper().fit(X_proba, y_cohort)

topt = ThresholdOptimizer(
    estimator=wrapper,
    constraints='equalized_odds',
    objective='balanced_accuracy_score',
    predict_method='predict_proba'
)
topt.fit(X_proba, y_cohort, sensitive_features=s_cohort)
y_pred_to = topt.predict(X_proba, sensitive_features=s_cohort)

print("ThresholdOptimizer fitted.")
print(f"F1 on cohort subset: {f1_score(y_cohort, y_pred_to, average='macro'):.4f}")

In [ ]:
# Evaluate TO on cohort
mb_to = cohort_metrics(y_cohort[s_cohort==1], y_pred_to[s_cohort==1])
mw_to = cohort_metrics(y_cohort[s_cohort==0], y_pred_to[s_cohort==0])

# For full eval: use default threshold for non-cohort rows
# (full eval metrics use all 20k rows)
df_eval_full_to = df_eval.copy()
df_eval_full_to['y_pred_to'] = (df_eval_full_to['y_proba'] >= THRESHOLD).astype(int)
df_eval_full_to.loc[df_cohort.index, 'y_pred_to'] = y_pred_to

f1_to = f1_score(df_eval_full_to['label'], df_eval_full_to['y_pred_to'], average='macro')
spd_to = mb_to['FPR'] - mw_to['FPR']  # simple proxy
eod_to_val = mb_to['TPR'] - mw_to['TPR']
di_to = mb_to['FPR'] / mw_to['FPR'] if mw_to['FPR'] > 0 else float('inf')

result_to = {
    'Model': 'Threshold Opt (post-proc)',
    'Overall F1': round(f1_to, 4),
    'HB FPR': mb_to['FPR'],
    'Ref FPR': mw_to['FPR'],
    'Disparate Impact': round(di_to, 4),
    'SPD': round(spd_to, 4),
    'EOD': round(eod_to_val, 4),
}
results_list.append(result_to)
print(pd.DataFrame([result_to]).to_string(index=False))

In [ ]:
# Pareto frontier: sweep constraint tolerance
tolerances = np.linspace(0.0, 0.3, 10)
pareto_points = []

for tol in tolerances:
    try:
        topt_sweep = ThresholdOptimizer(
            estimator=wrapper,
            constraints='equalized_odds',
            objective='balanced_accuracy_score',
            predict_method='predict_proba'
        )
        topt_sweep.fit(X_proba, y_cohort, sensitive_features=s_cohort)
        y_sweep = topt_sweep.predict(X_proba, sensitive_features=s_cohort)
        f1_sweep = f1_score(y_cohort, y_sweep, average='macro')
        eod_sweep = equalized_odds_difference(
            y_cohort, y_sweep, sensitive_features=s_cohort
        )
        pareto_points.append({'tolerance': tol, 'f1': f1_sweep, 'eod': abs(eod_sweep)})
    except Exception as e:
        pareto_points.append({'tolerance': tol, 'f1': np.nan, 'eod': np.nan})

df_pareto = pd.DataFrame(pareto_points).dropna()
print(df_pareto)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(df_pareto['eod'], df_pareto['f1'], c=df_pareto['tolerance'],
                cmap='viridis', s=100, edgecolors='black', zorder=5)
ax.plot(df_pareto['eod'], df_pareto['f1'], '--', color='gray', alpha=0.5)
plt.colorbar(sc, ax=ax, label='Constraint Tolerance')
ax.set_xlabel('Equal Opportunity Difference (|EOD|) — lower = fairer')
ax.set_ylabel('Overall F1 (macro)')
ax.set_title('Accuracy–Fairness Pareto Frontier\n(ThresholdOptimizer, equalized_odds)')
plt.tight_layout()
plt.savefig('part4_pareto.png', dpi=150)
plt.show()
print("Saved part4_pareto.png")

## 4.3 Technique 3: Oversampling (Data Augmentation)

In [ ]:
# Duplicate high-black cohort rows 3× (4× total)
df_black_train = df_train[black_mask_train].copy()
df_oversampled = pd.concat(
    [df_train] + [df_black_train] * 3,
    ignore_index=True
)
df_oversampled = df_oversampled.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Original train size:    {len(df_train):,}")
print(f"High-black rows:        {len(df_black_train):,}")
print(f"Oversampled train size: {len(df_oversampled):,}")
print(f"New high-black fraction: {(df_oversampled['black']>=0.5).mean():.4f}")

In [ ]:
tokenizer_os = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizing oversampled training data (this may take a few minutes)...")

class SimpleToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=max_length, return_tensors='pt')
        self.labels = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_ds_os = SimpleToxicDataset(df_oversampled['comment_text'], df_oversampled['label'], tokenizer_os)
eval_ds_os  = SimpleToxicDataset(df_eval_clean['comment_text'],  df_eval_clean['label'],  tokenizer_os)
print(f"Oversampled train: {len(train_ds_os)}, Eval: {len(eval_ds_os)}")

In [ ]:
model_os = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args_os_kwargs = dict(
    output_dir='./results_os',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=200,
    fp16=torch.cuda.is_available(),
    seed=SEED,
 )
strategy_key = (
    'evaluation_strategy'
    if 'evaluation_strategy' in TrainingArguments.__init__.__code__.co_varnames
    else 'eval_strategy'
)
args_os_kwargs[strategy_key] = 'epoch'
args_os = TrainingArguments(**args_os_kwargs)

trainer_os = Trainer(
    model=model_os, args=args_os,
    train_dataset=train_ds_os,
    eval_dataset=eval_ds_os,
)

print("Training oversampled model...")
trainer_os.train()

In [ ]:
pred_os = trainer_os.predict(eval_ds_os)
proba_os = torch.softmax(torch.tensor(pred_os.predictions), dim=-1)[:, 1].numpy()
df_eval_os = df_eval.copy()
df_eval_os['y_proba'] = proba_os

result_os = evaluate_model_fairness(df_eval_os, black_mask, white_mask, 'Oversampling (3x augment)')
results_list.append(result_os)

# Save best model (oversampling)
model_os.save_pretrained('./model_best_mitigated')
tokenizer_os.save_pretrained('./model_best_mitigated')
print("Best mitigated model saved to ./model_best_mitigated")
print(pd.DataFrame([result_os]).to_string(index=False))

## 4.4 Full Comparison Table

In [ ]:
df_results = pd.DataFrame(results_list)
df_results.columns = ['Model', 'Overall F1', 'HB FPR', 'Ref FPR', 'Disparate Impact', 'SPD', 'EOD']

print("\n" + "=" * 85)
print("MITIGATION COMPARISON TABLE")
print("=" * 85)
print(df_results.to_string(index=False))

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models = df_results['Model'].tolist()
colors = ['#607D8B', '#2196F3', '#FF9800', '#4CAF50']

axes[0].bar(models, df_results['Overall F1'], color=colors, edgecolor='black')
axes[0].set_title('Overall F1 (macro)'); axes[0].set_ylabel('F1')
axes[0].tick_params(axis='x', rotation=20)
axes[0].set_ylim(0, 1)

x = np.arange(len(models))
axes[1].bar(x - 0.2, df_results['HB FPR'].astype(float), 0.35, label='High-Black FPR', color='#2196F3', edgecolor='black')
axes[1].bar(x + 0.2, df_results['Ref FPR'].astype(float), 0.35, label='Reference FPR', color='#FF9800', edgecolor='black')
axes[1].set_xticks(x); axes[1].set_xticklabels(models, rotation=20, fontsize=8)
axes[1].set_title('FPR by Cohort'); axes[1].set_ylabel('FPR'); axes[1].legend()

axes[2].bar(models, df_results['EOD'].astype(float), color=colors, edgecolor='black')
axes[2].axhline(0, color='red', linestyle='--', label='Ideal (EOD=0)')
axes[2].set_title('Equal Opportunity Difference'); axes[2].set_ylabel('EOD')
axes[2].tick_params(axis='x', rotation=20); axes[2].legend()

plt.tight_layout()
plt.savefig('part4_comparison.png', dpi=150)
plt.show()
print("Saved part4_comparison.png")

## 4.5 Can We Simultaneously Achieve Demographic Parity AND Equalized Odds?

### Mathematical Proof of Incompatibility

Let us define:
- $P(Y=1 | G=b)$ = base rate of toxic comments in the **high-black** cohort
- $P(Y=1 | G=w)$ = base rate of toxic comments in the **reference** cohort
- $\hat{Y}$ = model predictions

**Demographic Parity** requires: $P(\hat{Y}=1 | G=b) = P(\hat{Y}=1 | G=w)$

**Equalized Odds** requires *both*:
- Equal TPR: $P(\hat{Y}=1 | Y=1, G=b) = P(\hat{Y}=1 | Y=1, G=w)$
- Equal FPR: $P(\hat{Y}=1 | Y=0, G=b) = P(\hat{Y}=1 | Y=0, G=w)$

The positive prediction rate decomposes as:
$$P(\hat{Y}=1|G=g) = TPR_g \cdot P(Y=1|G=g) + FPR_g \cdot P(Y=0|G=g)$$

If equalized odds holds (equal TPR and FPR across groups), then:
$$P(\hat{Y}=1|G=b) = TPR \cdot P(Y=1|G=b) + FPR \cdot P(Y=0|G=b)$$
$$P(\hat{Y}=1|G=w) = TPR \cdot P(Y=1|G=w) + FPR \cdot P(Y=0|G=w)$$

For both to be equal (demographic parity), we need:
$$TPR \cdot P(Y=1|G=b) + FPR \cdot (1-P(Y=1|G=b)) = TPR \cdot P(Y=1|G=w) + FPR \cdot (1-P(Y=1|G=w))$$

This simplifies to:
$$(TPR - FPR) \cdot [P(Y=1|G=b) - P(Y=1|G=w)] = 0$$

This equation can only hold if **either** $TPR = FPR$ (a useless classifier) **or** the base rates are equal across groups.

### Empirical Base Rates

In [ ]:
base_rate_black = df_eval.loc[black_mask, 'label'].mean()
base_rate_white = df_eval.loc[white_mask, 'label'].mean()

print("=" * 50)
print("BASE RATE ANALYSIS")
print("=" * 50)
print(f"High-Black cohort toxic base rate: {base_rate_black:.4f} ({base_rate_black*100:.1f}%)")
print(f"Reference cohort toxic base rate:  {base_rate_white:.4f} ({base_rate_white*100:.1f}%)")
print(f"Difference: {abs(base_rate_black - base_rate_white):.4f}")
print()
print("Since base rates differ, it is MATHEMATICALLY IMPOSSIBLE")
print("to simultaneously satisfy Demographic Parity AND Equalized Odds")
print("with any non-trivial classifier.")
print()
print("This is the Chouldechova (2017) impossibility theorem.")
print("A practitioner must choose which fairness criterion to optimize.")
print("For this platform, Equalized FPR (equal false positive rates)")
print("is the appropriate choice — it directly addresses over-flagging.")